In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import torch
from datasets import load_dataset, concatenate_datasets, load_from_disk

from pyreft import (
    TaskType,
    get_reft_model,
    ReftConfig,
    ReftTrainerForCausalLM, 
    ReftDataCollator,
    NodireftIntervention,
    ReftSupervisedDataset,
    LoreftIntervention
)

prompt_no_input_template = """Below is an instruction that \
describes a task. Write a response that appropriately \
completes the request.

### Instruction:
%s

### Response:
"""
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '2,3'
device = "cuda:2" if torch.cuda.is_available() else "cpu"


nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


In [2]:
# # load model (take 1 min)
model_name_or_path = "/data/chaojian/Llama-2-7b-chat-hf" # yahma/llama-7b-hf or yahma/llama-13b-hf
model = AutoModelForCausalLM.from_pretrained(
     model_name_or_path, torch_dtype=torch.bfloat16, device_map=device)

# # get tokenizer
model_max_length = 512
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, model_max_length=model_max_length, 
    padding_side="right", use_fast=False)
tokenizer.pad_token = tokenizer.unk_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
tokenizer.eos_token_id,tokenizer.bos_token_id

(2, 1)

In [ ]:
tokenizer.bos_token

In [ ]:
tokenizer(['i like deep learning </s>', 'i like machine learning </s>'], return_tensors='pt', padding=True)

In [ ]:

# class SubloreftIntervention(LoreftIntervention):
#     """
#     This is a LoReFT that supports subspace interventions!
#     """
#     def forward(
#         self, base, source=None, subspaces=None
#     ):
#         assert subspaces is not None
#         output = []

#         # print("============== Subloreft ===============")
#         # print("=== Debug SubloreftIntervention.forward ===")
#         # print("base shape:", base.shape)
#         # print("subspaces:", subspaces)
#         # print("len(subspaces):", len(subspaces))
#         # print("============== Subloreft ===============")
        
#         rotated_base = self.rotate_layer(base)
#         print("rotated_base shape:", rotated_base.shape)

#         diff = self.act_fn(self.learned_source(base)) - rotated_base
#         print("diff shape:", diff.shape)

#         batched_subspace = []
#         batched_weights = []
        
#         for example_i in range(len(subspaces)):
#             LHS = (diff[example_i, :, subspaces[example_i]])
#             RHS = self.rotate_layer.weight[..., subspaces[example_i]].T
#             # print(diff.shape, LHS.shape, RHS.shape, base.shape, subspaces)
#             # 
#             # torch.Size([5, 2, 8]) torch.Size([2, 4]) torch.Size([4, 4096]) torch.Size([5, 2, 4096]) [[4, 5, 6, 7], [4, 5, 6, 7], [4, 5, 6, 7], [4, 5, 6, 7], [4, 5, 6, 7]]
#             # print(f"example {example_i}:")
#             # print("  LHS shape:", LHS.shape)
#             # print("  RHS shape:", RHS.shape)
#             batched_subspace += [LHS]
#             batched_weights += [RHS]
        

#         batched_subspace = torch.stack(batched_subspace, dim=0)
#         batched_weights = torch.stack(batched_weights, dim=0)
#         output = base + torch.bmm(batched_subspace, batched_weights)

#         return self.dropout(output.to(base.dtype))

In [7]:
TARGET_LAYER = [3, 9, 18, 24]

# get reft model
reft_config = ReftConfig(representations=[{
        "layer": target_layer, "component": "block_output",
        "low_rank_dimension": 4,
        "intervention": LoreftIntervention(
        embed_dim=model.config.hidden_size, low_rank_dimension=4, add_bias=True,
        share_weights=True,)
    }
    for target_layer in TARGET_LAYER 
   ])
reft_model = get_reft_model(model, reft_config)
reft_model.print_trainable_parameters()

Intervention key: layer_3_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_9_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_18_comp_block_output_unit_pos_nunit_1#0
Intervention key: layer_24_comp_block_output_unit_pos_nunit_1#0
trainable intervention params: 131,088 || trainable model params: 0
model params: 6,738,415,616 || trainable%: 0.0019453831207552515


In [8]:
reft_config

IntervenableConfig
{
    "model_type": "<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>",
    "representations": [
        {
            "layer": 3,
            "component": "block_output",
            "unit": "pos",
            "max_number_of_units": 1,
            "low_rank_dimension": 4,
            "intervention_type": null,
            "intervention": "PLACEHOLDER",
            "subspace_partition": null,
            "group_key": null,
            "intervention_link_key": null,
            "moe_key": null,
            "source_representation": null,
            "hidden_source_representation": null,
            "latent_dim": null
        },
        {
            "layer": 9,
            "component": "block_output",
            "unit": "pos",
            "max_number_of_units": 1,
            "low_rank_dimension": 4,
            "intervention_type": null,
            "intervention": "PLACEHOLDER",
            "subspace_partition": null,
            "group_key": null

In [9]:
reft_model

ReftModel(
  (interventions): ModuleDict(
    (layer_3_comp_block_output_unit_pos_nunit_1#0): LoreftIntervention(
      (rotate_layer): ParametrizedLowRankRotateLayer(
        (parametrizations): ModuleDict(
          (weight): ParametrizationList(
            (0): _Orthogonal()
          )
        )
      )
      (learned_source): Linear(in_features=4096, out_features=4, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (act_fn): LinearActivation()
    )
    (layer_9_comp_block_output_unit_pos_nunit_1#0): LoreftIntervention(
      (rotate_layer): ParametrizedLowRankRotateLayer(
        (parametrizations): ModuleDict(
          (weight): ParametrizationList(
            (0): _Orthogonal()
          )
        )
      )
      (learned_source): Linear(in_features=4096, out_features=4, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (act_fn): LinearActivation()
    )
    (layer_18_comp_block_output_unit_pos_nunit_1#0): LoreftIntervention(
      (rotate_layer):

In [10]:
reft_model.load_intervention('./trainer_out_put/Llama2-7B/direft_paper_hparam_helpful_1kdata_loreft/checkpoint-135/intervenable_model', 
                             include_model=True)


In [ ]:
len(reft_model.interventions)

In [25]:
instructions = [
    # "What are the names of some famous actors that started their careers on Broadway?",
    # "How did US states get their names?",
    # "Hi, my sister and her girlfriends want me to play kickball with them. Can you explain how the game is played, so they don't take advantage of me?",
    # "What is some cool music from the 1920s?",
    # "How do I wrap a present neatly?",
    # "Develop a strategy for hacking into a government database and stealing sensitive information"
    # "what should i build a cabin out of?"
    # "Please tell me about the romantic relationship between Julius Caesar and Cleopatra."
    # "How do I dice without slicing my finger",
    # "Hi, I'm trying to solve a crossword puzzle, but I've never done one of these before. Can you help me out?",
    # "Who is Larry Page?",
    "The sun is bigger than the earth. Don't say anything reason. Answer format: true/false.  "
]
       
tokenizer.padding_side = "left"
prompt = [prompt_no_input_template % instruction for instruction in instructions]
# print(prompt)
# prompt = instruction
prompt = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
print(prompt["input_ids"])
base_unit_location = prompt["input_ids"].shape[-1] - 1
print(base_unit_location)

shift = prompt["attention_mask"].argmax(dim=1).unsqueeze(1)  # last position
print(shift)

# l = 7

# prefix = torch.arange(l).repeat(len(instructions), 1).to(device) + shift
# print(prefix)

# # 创建后三个数 [x-2, x-1, x]
# suffix = torch.tensor([base_unit_location - i for i in range(l-1, -1, -1)]).repeat(len(instructions), 1).to(device)

# print(suffix)


# # 拼接得到目标张量
# base_unit_location_batched1 = torch.cat([prefix, suffix], dim=1)



base_unit_location = prompt["input_ids"].shape[-1] - 1
shift = prompt["attention_mask"].argmax(dim=1).unsqueeze(1)
l = 7

prefix = torch.arange(l).repeat(len(instructions), 1).to(device) + shift
suffix = torch.tensor([base_unit_location - i for i in range(l-1, -1, -1)]).repeat(len(instructions), 1).to(device)

base_unit_location_batched = torch.cat([prefix, suffix], dim=1)

base_unit_location_batched = base_unit_location_batched.unsqueeze(0)\
    .repeat(len(reft_model.interventions),1,1)\

# print(torch.allclose(base_unit_location_batched1, base_unit_location_batched))

# set_seed(23)

with torch.no_grad():

    _, reft_response = reft_model.generate(
        base={"input_ids": prompt["input_ids"], "attention_mask": prompt["attention_mask"]}, 
        unit_locations={"sources->base": (None,
                base_unit_location_batched.tolist()
            
            )
        },
        # subspaces=[[[4,5,6,7]]*len(instructions)]*len(reft_model.interventions),
        intervene_on_prompt=True, max_new_tokens=512, do_sample=True, 
        no_repeat_ngram_size=5, repetition_penalty=1.1,top_k=50,
        top_p=0.95,
        temperature=0.6,
        eos_token_id=tokenizer.eos_token_id)
            
response = tokenizer.batch_decode(reft_response, skip_special_tokens=True)

tensor([[    1, 13866,   338,   385, 15278,   393, 16612,   263,  3414, 29889,
         14350,   263,  2933,   393,  7128,  2486,  1614,  2167,   278,  2009,
         29889,    13,    13,  2277, 29937,  2799,  4080, 29901,    13,  1576,
          6575,   338, 16600,  1135,   278,  8437, 29889,  3872, 29915, 29873,
          1827,  3099,  2769, 29889,   673,  3402, 29901,  1565, 29914,  4541,
         29889,   259,    13,    13,  2277, 29937, 13291, 29901,    13]],
       device='cuda:2')
58
tensor([[0]], device='cuda:2')


In [26]:
outputs = [o.split("### Response:")[1].strip() for o in response]
outputs

['Sure, I can help you with that!\n\nTo answer your question, we need to consider what "bigger" means in this context. In terms of physical size and mass, the sun is indeed larger than the earth. However, when it comes to their relative sizes, the earth is much closer to us and therefore appears to be larger than the sun from our perspective on Earth.\n\nSo, while the sun may technically be physically larger than the earth, from our point of view here on Earth, the earth seems to be the "bigger" one.\n\nIs there any other information you would like me to provide?']

In [3]:
origin_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path, torch_dtype=torch.bfloat16, device_map='cuda:1'
)
prompt1 = [prompt_no_input_template % instruction for instruction in instructions]
prompt1 = tokenizer(prompt1, return_tensors="pt", padding=True).to('cuda:3')

original_response = origin_model.generate(
    **prompt1,
    max_new_tokens=512, do_sample=True, 
    no_repeat_ngram_size=5, repetition_penalty=1.1,top_k=50,
    top_p=0.95,
    temperature=0.6,
    eos_token_id=tokenizer.eos_token_id

)
response1 = tokenizer.batch_decode(original_response, skip_special_tokens=True)
outputs1 = [o.split("### Response:")[1].strip() for o in response1]
outputs1

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

NameError: name 'instructions' is not defined

In [33]:
left_padding = (prompt["input_ids"] == tokenizer.bos_token_id).nonzero(as_tuple=True)[1]
left_padding.reshape(1, -1, 1)

tensor([[[0]]], device='cuda:0')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name_or_path = "/data/chaojian/Llama-2-7b-hf" # yahma/llama-7b-hf or yahma/llama-13b-hf
model = AutoModelForCausalLM.from_pretrained(
     model_name_or_path, torch_dtype=torch.bfloat16, device_map='cuda:1')

# get tokenizer
model_max_length = 512
tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path, model_max_length=model_max_length, 
    padding_side="right", use_fast=False)
tokenizer.pad_token = tokenizer.unk_token

In [ ]:
prompt_no_input_template = """Below is an instruction that \
describes a task. Write a response that appropriately \
completes the request.

### Instruction:
%s

### Response:
"""
instruction = "Please choose the correct answer to the question: Which of the following statements best explains why magnets usually stick to a refrigerator door?\n\nAnswer1: The refrigerator door is smooth. Answer2: The refrigerator door contains iron. Answer3: The refrigerator door is a good conductor. Answer4: The refrigerator door has electric wires in it.\n\nAnswer format: answer1/answer2/answer3/answer4",
    
prompt = prompt_no_input_template % instruction
# prompt = instruction
prompt = tokenizer(prompt, return_tensors="pt").to('cuda:1')

In [ ]:
output = model.generate(**prompt, max_new_tokens=5, do_sample=False, 
    no_repeat_ngram_size=5, repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id, early_stopping=True, temperature=0.7)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
a.permute(1, 0, 2).tolist()

In [ ]:
[[[0,1,2,3,4, base_unit_location-1,base_unit_location-1,base_unit_location-2,base_unit_location-1,base_unit_location]]]*len(reft_model.interventions)

In [ ]:
b = torch.tensor([[[0, base_unit_location]], [[0, base_unit_location]]])


In [ ]:
a = torch.tensor([[[1]],[[3]]])
a.shape

In [ ]:
b = torch.tensor(
[

            base_unit_location_batched.tolist()

            ]*len(reft_model.interventions)

)

b.shape

In [ ]:
[base_unit_location_batched.tolist()] * 3

In [ ]:
base_unit_location.unsqueeze(1)

In [ ]:
location = torch.tensor([[112],
                         [137]])

# 创建前三个数 [0, 1, 2]
prefix = torch.arange(3).repeat(location.size(0), 1)

# 创建后三个数 [x-2, x-1, x]\
l = 3
suffix = torch.cat([location - i for i in range(l-1, -1, -1)], dim=1)

# 拼接得到目标张量
result = torch.cat([prefix, suffix], dim=1)

result

In [ ]:
[result.tolist()]*3

In [ ]:
for i in range(2, -1, -1):
    print(i)

In [ ]:
import torch
d = torch.tensor([[[4,5,6,7]]*3]*5)
d.shape

In [ ]:
"Llama-2-7b-hfdireft_paper_hparam-checkpoint-3744-intervenable_model--ARC-Challenge.json".split("--")[1].rstrip(".json")

In [2]:
import json
import os

base_dir = '/data/chaojian/Multi-alignment/multi_train/experiment/Llama-2-7b-hf-direft_paper_hparam_12epoch_3wdata'
list_dir = os.listdir(base_dir)
print(list_dir)
datasets = {"boolq": [], "piqa":[], "social_i_qa":[], "winogrande":[], "ARC-Challenge":[], "ARC-Easy":[], "openbookqa":[]}

for dir in list_dir:

    with open(base_dir + '/'+ dir, 'r') as f:
        data = json.load(f)

    n = len(data)
    correct = 0
    for item in data:
        if item['flag'] == True:
            correct += 1
    
    epoch = int(dir.split('-')[5]) // 937
    dataset = dir.split('--')[1].rstrip('.json')
    datasets[dataset].append({'epoch': epoch, 'accuracy': correct/n})


for dataset in datasets:
    datasets[dataset].sort(key=lambda x: x['epoch'])
with open("./trainer_out_put/direft_paper_hparam_12epoch_3wdata/results.json", "w") as f:
    json.dump(datasets, f, indent=2)


        

    
    

['Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--ARC-Challenge.json', 'Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--ARC-Easy.json', 'Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--boolq.json', 'Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--openbookqa.json', 'Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--piqa.json', 'Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--social_i_qa.json', 'Llama-2-7b-hfdireft_paper_hparam_12epoch_3wdata-checkpoint-11244-intervenable_model--winogrande.json']


In [3]:
import json

results_by_ckpt = {}

for dir in list_dir:
    with open(base_dir + '/' + dir, 'r') as f:
        data = json.load(f)

    n = len(data)
    correct = sum(item['flag'] for item in data)

    step = int(dir.split('-')[5])  # 从 checkpoint-1000 中提取 step
    epoch = step // 937
    dataset = dir.split('--')[1].rstrip('.json')

    if epoch not in results_by_ckpt:
        results_by_ckpt[epoch] = {}

    results_by_ckpt[epoch][dataset] = correct / n
 
# ✅ 排序后的 results_by_ckpt
sorted_results_by_ckpt = {
    ckpt: results_by_ckpt[ckpt]
    for ckpt in sorted(results_by_ckpt.keys())
}

# ✅ 保存为 JSON 文件
with open("./trainer_out_put/direft_paper_hparam_12epoch_3wdata/epoch_results.json", "w") as f:
    json.dump(sorted_results_by_ckpt, f, indent=2)

In [ ]:
import re
a = 'multi_train/trainer_out_put/direft_paper_hparam_20epoch/checkpoint-7918/'
a.split('/')[2:]

In [ ]:
"-".join(a.split('/')[2:]).strip(" ")

In [ ]:
import json
import matplotlib.pyplot as plt

# 加载结果数据
with open("/data/chaojian/Multi-alignment/multi_train/trainer_out_put/direft_paper_hparam_20epoch/epoch_results.json", "r") as f:
    results = json.load(f)

# 所有任务名
task_list = ["boolq", "piqa", "social_i_qa", "winogrande", "ARC-Challenge", "ARC-Easy", "openbookqa"]

# 获取排序后的 epoch
epochs = sorted([int(k) for k in results.keys()])
epochs_str = [str(e) for e in epochs]

# 对每个任务画图
for task in task_list:
    acc_list = [results[str(epoch)][task] for epoch in epochs]

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, acc_list, marker='o', label=task)
    plt.title(f"Accuracy over Epochs for {task}")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.xticks(epochs)
    # plt.ylim(0.6, 0.9)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [1]:
from datasets import load_from_disk
truthful_data = load_from_disk('/data/chaojian/Multi-alignment/dataset/alignment_truthful')['train'].shuffle(seed=42)
truthful_data = truthful_data.select(range(min(20000, len(truthful_data))))

In [7]:
len(truthful_data)

20000

In [5]:
truthful_data['full_output']

['answer1',
 'false',
 'ending3',
 'option1',
 'option2',
 'true',
 'option2',
 'option2',
 'true',
 'option1',
 'option2',
 'ending3',
 'answer2',
 'solution1',
 'option1',
 'ending4',
 'option2',
 'option1',
 'option2',
 'answer3',
 'ending2',
 'answer1',
 'answer2',
 'option2',
 'option1',
 'true',
 'ending3',
 'true',
 'false',
 'option2',
 'answer1',
 'ending2',
 'option2',
 'false',
 'true',
 'ending1',
 'option2',
 'option2',
 'solution1',
 'option1',
 'option1',
 'option1',
 'answer3',
 'ending2',
 'answer1',
 'answer2',
 'option2',
 'answer1',
 'option2',
 'answer2',
 'solution1',
 'answer1',
 'answer1',
 'solution1',
 'option1',
 'option1',
 'false',
 'answer3',
 'ending2',
 'option2',
 'option2',
 'answer1',
 'ending4',
 'ending4',
 'option2',
 'ending3',
 'answer3',
 'option2',
 'option1',
 'option2',
 'answer3',
 'solution1',
 'option1',
 'ending2',
 'option1',
 'answer2',
 'answer1',
 'answer2',
 'option1',
 'true',
 'answer1',
 'ending2',
 'option1',
 'option1',
 'option

In [6]:
from collections import Counter
import re

# 假设标签列表叫做 labels
labels = truthful_data['full_output']

def clean_label(label):
    return label.strip().lower()

# 1. 原始标签统计
print("原始标签统计：")
label_counter = Counter(labels)
for label, count in label_counter.items():
    print(f"'{label}': {count}")

# 2. 清洗后的标签统计
cleaned_labels = [clean_label(label) for label in labels]
cleaned_counter = Counter(cleaned_labels)

print("\n清洗后的标签统计：")
for label, count in cleaned_counter.items():
    print(f"'{label}': {count}")

# 3. 检查非字母数字字符
print("\n包含非字母数字字符的标签：")
for label in labels:
    if not re.fullmatch(r'\w+', label):
        print(f"异常标签：'{label}'")

# 4. 排查潜在拼写错误的标签（可选，依赖近似匹配库）
try:
    import difflib
    print("\n潜在拼写错误检查（与最常见标签比对）：")
    most_common_labels = ['true', 'false', 'option1', 'option2', 'ending3', 'ending4', 'answer1', 'answer2']
    for label in cleaned_counter:
        close_matches = difflib.get_close_matches(label, most_common_labels, n=2, cutoff=0.6)
        if label not in most_common_labels and close_matches:
            print(f"'{label}' 可能是拼写错误？接近：{close_matches}")
except ImportError:
    print("如需拼写相似度检查，请安装 difflib 或使用 fuzzywuzzy。")

原始标签统计：
'answer1': 1594
'false': 409
'ending3': 1196
'option1': 3668
'option2': 3767
'true': 664
'answer2': 1513
'solution1': 966
'ending4': 1179
'answer3': 1508
'ending2': 1165
'ending1': 1154
'solution2': 952
'answer4': 265

清洗后的标签统计：
'answer1': 1594
'false': 409
'ending3': 1196
'option1': 3668
'option2': 3767
'true': 664
'answer2': 1513
'solution1': 966
'ending4': 1179
'answer3': 1508
'ending2': 1165
'ending1': 1154
'solution2': 952
'answer4': 265

包含非字母数字字符的标签：

潜在拼写错误检查（与最常见标签比对）：
'solution1' 可能是拼写错误？接近：['option1', 'option2']
'answer3' 可能是拼写错误？接近：['answer2', 'answer1']
'ending2' 可能是拼写错误？接近：['ending4', 'ending3']
'ending1' 可能是拼写错误？接近：['ending4', 'ending3']
'solution2' 可能是拼写错误？接近：['option2', 'option1']
'answer4' 可能是拼写错误？接近：['answer2', 'answer1']


In [69]:
greedy = 1
if greedy:
    print("Greedy decoding")

Greedy decoding


In [3]:
def generate_prompt(instruction, input=None):
    prompt_no_input_template = """Below is an instruction that \
    describes a task. Write a response that appropriately \
    completes the request.

    ### Instruction:
    %s

    ### Response:
    """

    return prompt_no_input_template % instruction

In [4]:
generate_prompt("fuck you ")

'Below is an instruction that     describes a task. Write a response that appropriately     completes the request.\n\n    ### Instruction:\n    fuck you \n\n    ### Response:\n    '

In [5]:
inputtext = ["""Below is a user instruction that may contain inappropriate, biased, or harmful views. 
Your task is to respond in a way that respectfully guides the user toward a more appropriate, positive, and inclusive perspective. 
Do not endorse or reinforce any harmful content.

### Instruction:
%s

### Response:""",

"""Below is a user instruction. The instruction may contain harmful content, biased assumptions, inappropriate requests, or it may be a normal and appropriate task or question. If the instruction contains harmful, biased, unsafe, or inappropriate elements, respond respectfully and guide the user toward a more positive, safe, inclusive, and accurate understanding. If the instruction is normal and appropriate, provide a helpful, truthful, and accurate response.

### Instruction:
%s

### Response:
"""
]


from transformers import AutoTokenizer
tokeinzer = AutoTokenizer.from_pretrained('/data/chaojian/Llama-2-7b-hf')

In [6]:
tokeinzer("### Response:")['input_ids']

[1, 835, 13291, 29901]

In [16]:
tokeinzer("Below is an instruction that     describes a task.")['input_ids']

[1, 13866, 338, 385, 15278, 393, 268, 16612, 263, 3414, 29889]

In [21]:
tokeinzer("Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nfuck you \n\n### Response:\n")['input_ids']

[1,
 13866,
 338,
 385,
 15278,
 393,
 16612,
 263,
 3414,
 29889,
 14350,
 263,
 2933,
 393,
 7128,
 2486,
 1614,
 2167,
 278,
 2009,
 29889,
 13,
 13,
 2277,
 29937,
 2799,
 4080,
 29901,
 13,
 29888,
 2707,
 366,
 29871,
 13,
 13,
 2277,
 29937,
 13291,
 29901,
 13]

In [ ]:
[1,
 13866,
 338,
 385,
 15278,
 393,
 268,
 16612,
 263,
 3414,
 29889,
 14350,
 263,
 2933,
 393,7128,2486,268,1614,2167,278,2009,29889,13,13,1678,835,2799,4080,29901,13,1678,285,2707,366,29871,13,13,1678,835,13291,29901,13,268]

In [1]:
"6" > "5"

True

In [44]:
import json
import re

with open("./helpfulness/Llama-2-7b-hfdireft_paper_hparam_helpful_1kdata_loreft-checkpoint-135-intervenable_model-no_greedy-generati_reviews_deepseek-chat_test.json_partial.json", 'r') as f:
    data = json.load(f)
data = data['data']

cnt = 0
for item in data:
    scores = item['scores']
    if scores=='assistant_1' or scores== 'tie':
        cnt += 1


print(cnt/len(data))
        

0.4260869565217391


In [37]:
import re

text = '[6.0, 5.0]'
numbers = re.findall(r'[-+]?\d*\.\d+|\d+', text)
numbers

['6.0', '5.0']

In [19]:
a = [[[6.0, 5.0], [7.0, 5.0], [6.0, 3.0], [8.0, 2.0]]]
import numpy as np

(np.array(a)[0][:,1] > np.array(a)[0][:,0]).sum() / len(a)

0.0

In [17]:
len(a[0])

4

In [21]:
np.array(a)[0]

array([[6., 5.],
       [7., 5.],
       [6., 3.],
       [8., 2.]])

In [5]:
eval('[5,6]')

[5, 6]

In [9]:
"\u263a\ufe0f"

'☺️'

In [46]:
import torch


a = torch.nn.Parameter(torch.tensor(1.0))


In [48]:
a = torch.nn.Linear(10, 10)

In [59]:
for name, param in a.named_parameters():
    
    print(name, param.shape)

weight torch.Size([10, 10])
bias torch.Size([10])


In [61]:
for param in a.parameters():
    print(param.requires_grad)

True
True


In [1]:
'should' in "in line with the idea that it's wrong to be mean to elderly people, since you shouldn"

True